# Session 2: Types of Data & Measurement Scales

Covers Numerical vs Categorical classification, measurement scales (Nominal/Ordinal/Interval/Ratio), time-series identification, and a simple rule-based scale detector.

## 1. Classifying Zomato Dataset Columns

| Column | Type | Sub-type | Reasoning |
|---|---|---|---|
| Restaurant Name | Categorical | Nominal | Just labels/names, no order or numeric meaning |
| Cuisine Type | Categorical | Nominal | Categories like Chinese, Italian, North Indian — no natural ranking |
| Average Cost for Two | Numerical | Continuous | A measured/calculated money value that can take any value in a range (e.g. ₹450.50) |
| User Rating | Numerical | Continuous | Typically a decimal value (e.g. 4.3) on a 0–5 scale, so it behaves like continuous data (though it's sometimes treated as Ordinal if shown only as whole stars) |
| City | Categorical | Nominal | City names have no inherent order |
| Open/Closed Status | Categorical | Nominal | Only two labels (Open/Closed) with no ranking between them |


In [ ]:
zomato_column_types = {
    'Restaurant Name': ('Categorical', 'Nominal'),
    'Cuisine Type': ('Categorical', 'Nominal'),
    'Average Cost for Two': ('Numerical', 'Continuous'),
    'User Rating': ('Numerical', 'Continuous'),
    'City': ('Categorical', 'Nominal'),
    'Open/Closed Status': ('Categorical', 'Nominal'),
}

for col, (main_type, sub_type) in zomato_column_types.items():
    print(f"{col:25s} -> {main_type} ({sub_type})")

## 2. Eight Real-Life Examples of Measurement Scales

| # | App | Data Example | Measurement Scale | Why |
|---|---|---|---|---|
| 1 | Instagram | Number of Followers | Ratio | Has a true zero (0 followers is meaningful) and ratios make sense (1000 followers is 2× 500) |
| 2 | Instagram | Time a Post was Uploaded (e.g. 5:30 PM) | Interval | Clock time has equal intervals, but '0:00' isn't an absolute absence of time, so ratios ('twice as late') don't make sense |
| 3 | Instagram | Post Language (English, Hindi, etc.) | Nominal | Just labels, no natural order |
| 4 | Flipkart | Product Price | Ratio | True zero (₹0) exists, and a ₹2000 item is genuinely 2× the price of a ₹1000 item |
| 5 | Flipkart | Customer Review Star Rating (1–5) | Ordinal | Higher numbers mean better reviews, but the *gap* between 3★ and 4★ isn't guaranteed equal to the gap between 4★ and 5★ |
| 6 | Flipkart | Payment Method (UPI, Card, COD) | Nominal | Categories with no inherent ranking |
| 7 | IRCTC | Train Class (Sleeper < AC3 < AC2 < AC1) | Ordinal | Classes have a clear order (comfort/price), but the difference between classes isn't a fixed numeric unit |
| 8 | IRCTC | Train Departure Time (e.g. 14:35) | Interval | Equal spacing between times, but there's no 'true zero' departure time that means 'no time at all' |


In [ ]:
examples = [
    ('Instagram', 'Number of Followers', 'Ratio'),
    ('Instagram', 'Time of Post', 'Interval'),
    ('Instagram', 'Post Language', 'Nominal'),
    ('Flipkart', 'Product Price', 'Ratio'),
    ('Flipkart', 'Review Star Rating (1-5)', 'Ordinal'),
    ('Flipkart', 'Payment Method', 'Nominal'),
    ('IRCTC', 'Train Class', 'Ordinal'),
    ('IRCTC', 'Train Departure Time', 'Interval'),
]

print(f"{'App':<12}{'Data Example':<28}{'Scale':<10}")
print('-' * 50)
for app, example, scale in examples:
    print(f"{app:<12}{example:<28}{scale:<10}")

## 3. Spotify Listening History: Time-Series vs Categorical vs Numerical

Columns: **Date**, **Song Name**, **Play Count per Day**

- **Date** → This is the **time-series component**. It's the index along which the data is ordered chronologically; each row represents a distinct point in time (a day).
- **Song Name** → **Categorical (Nominal)**. Song titles are labels with no numeric meaning or order.
- **Play Count per Day** → **Numerical (Discrete)**. It's a count (0, 1, 2, 3, ...) — you can't play a song 2.5 times, so it takes whole-number values only.

**Putting it together:** the dataset as a whole is a *time-series* dataset because it tracks how a numerical variable (Play Count) changes for each categorical entity (Song Name) across a time dimension (Date). Date itself acts as the time axis, Song Name groups the data, and Play Count per Day is the value being tracked over time.

In [ ]:
spotify_columns = {
    'Date': 'Time-series index (temporal)',
    'Song Name': 'Categorical - Nominal',
    'Play Count per Day': 'Numerical - Discrete',
}

for col, classification in spotify_columns.items():
    print(f"{col:22s} -> {classification}")

## 4. Auto-Detecting Measurement Scale from a Column of Values

**Logic overview:**

1. **Check data type first**
   - If values are strings/text → likely **Nominal** or **Ordinal**
   - If values are numbers → likely **Interval** or **Ratio**

2. **For text/string columns:**
   - If the unique values match a known ordered set (e.g. `Low, Medium, High` or `Pending, Shipped, Delivered`) → **Ordinal**
   - Otherwise (e.g. `UPI, Card, COD`) → **Nominal**

3. **For numeric columns:**
   - If `0` is a meaningful 'absence of the thing' (e.g. price, quantity, distance) and ratios make sense → **Ratio**
   - If the values can be negative, or `0` doesn't mean 'nothing' (e.g. a temperature-like or time-like scale) → **Interval**

This is a simplified heuristic, not a guaranteed-correct classifier — real-world data often needs domain context to fully resolve ambiguous cases.

In [ ]:
def detect_scale(column, known_ordinal_sets=None):
    """
    Very simple rule-based detector for measurement scale.
    column: list/array of values (all from one column)
    known_ordinal_sets: optional list of sets representing known ordered categories
                         e.g. [{'low','medium','high'}, {'pending','shipped','delivered'}]
    Returns: 'Nominal', 'Ordinal', 'Interval', or 'Ratio'
    """
    if known_ordinal_sets is None:
        known_ordinal_sets = [
            {'low', 'medium', 'high'},
            {'pending', 'shipped', 'delivered', 'cancelled'},
            {'sleeper', 'ac3', 'ac2', 'ac1'},
        ]

    values = [v for v in column if v is not None]
    if not values:
        return 'Unknown'

    # Rule 1: Are all values numeric?
    is_numeric = all(isinstance(v, (int, float)) for v in values)

    if not is_numeric:
        # Treat as text -> check against known ordinal sets
        unique_vals = {str(v).strip().lower() for v in values}
        for ordinal_set in known_ordinal_sets:
            if unique_vals.issubset(ordinal_set):
                return 'Ordinal'
        return 'Nominal'
    else:
        # Numeric -> decide Interval vs Ratio
        has_negative = any(v < 0 for v in values)
        has_true_zero_meaning = min(values) >= 0  # crude proxy: no negatives implies a meaningful zero lower bound

        if has_negative:
            # Negative values imply zero is just a reference point, not 'nothing'
            return 'Interval'
        elif has_true_zero_meaning:
            return 'Ratio'
        else:
            return 'Interval'


# --- Quick tests using Flipkart-style order history columns ---
test_columns = {
    'Order Amount': [499.0, 1200.5, 0.0, 3400.0],           # expect Ratio
    'Delivery Status': ['pending', 'shipped', 'delivered'],  # expect Ordinal
    'Payment Method': ['UPI', 'Card', 'COD', 'UPI'],         # expect Nominal
    'Temperature-like Score': [-5, 0, 10, -2],               # expect Interval
}

for col_name, col_data in test_columns.items():
    print(f"{col_name:25s} -> {detect_scale(col_data)}")